# Instruction fine-tuning on two GPUs

Configure the run, inspect the dataset and environment, and launch the standalone Accelerate training script. Training runs in separate processes; the model is saved to the output directory.


## 1. Setup and configuration
Run from the repository or a subdirectory of it. Change the settings below before launching.


In [1]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

project_root = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "model_training").is_dir()),
    None,
)

if project_root is None:
    raise RuntimeError("Run this notebook from inside the deep_learning repository")

In [2]:
file_name = project_root / "datasets/instruction_tuning/instruction-data.json"
output_dir = project_root / "execution_results/qwen_instruction"


In [3]:
# script = project_root / "model_training/post_training/02-sft/train_instruction.py"

In [4]:
from torch.utils.data import DataLoader, Subset
# from model_training.framework.data.post_train.dataloader import CyclingDataLoader
from datetime import datetime
from model_training.framework.train.tracking.logger import ExperimentLogger


In [5]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
num_gpus = 2
epochs = 2
batch_size = 1
accumulation_steps = 4
max_length = 1024
learning_rate = 1e-5
weight_decay = 0.1
print(f"[STAGE] setup | repository: {project_root} | Python: {sys.executable}")
print(f"✓ model: {model_name} | GPUs: {num_gpus} | output: {output_dir}")


[STAGE] setup | repository: /coc/scratch/vchopra/learning_env/Deep-Learning | Python: /coc/scratch/vchopra/learning_env/learning_env/bin/python
✓ model: Qwen/Qwen2.5-3B-Instruct | GPUs: 2 | output: /coc/scratch/vchopra/learning_env/Deep-Learning/execution_results/qwen_instruction


## 2. Check environment and data
The JSON file must already exist. The original instruction-tuning notebook downloads it. The script uses the first 85% for training.


In [6]:
! nvidia-smi

overwriting variable ['FWDLLM_USER', 'GOTOOLCHAIN']
/nethome/dgarg39/.bashrc: line 43: /nethome/dgarg39/.cargo/env: No such file or directory
Tue Nov 17 21:59:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:01:00.0 Off |                    0 |
|  0%   26C    P8             30W /  300W |       0MiB /  46068MiB |      0%      Default 

In [7]:
print("[STAGE] environment and dataset")
missing = [name for name in ("torch", "transformers", "accelerate") if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"Missing packages in the kernel environment: {', '.join(missing)}")
# if not script.is_file():
    # raise FileNotFoundError(f"Training script not found: {script}")
if not file_name.is_file():
    raise FileNotFoundError(f"Dataset not found: {file_name}; download it using the instruction-tuning notebook")
with file_name.open(encoding="utf-8") as f:
    data = json.load(f)
if not isinstance(data, list) or not data:
    raise ValueError("Dataset must be a non-empty JSON list")
import torch
print(f"✓ examples: {len(data)} | training split: {int(len(data) * 0.85)}")
print(f"✓ visible CUDA GPUs: {torch.cuda.device_count()}")
if torch.cuda.device_count() <= num_gpus:
    raise RuntimeError(f"Requested {num_gpus} GPUs but fewer are visible to this kernel")
print(f"✓ nominal effective batch size: {batch_size * num_gpus * accumulation_steps} examples")


[STAGE] environment and dataset
✓ examples: 1100 | training split: 935
✓ visible CUDA GPUs: 8
✓ nominal effective batch size: 8 examples


## 3. Load the dataset

In [ ]:
train_loader, val_loader, test_loader = return_dataset(data_dir, file_name, url)

## 4. Run training
This cell starts the actual training job and waits for it to finish. A failed worker raises an error in the notebook.

In [ ]:
text_to_test = "Every effort moves you"
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )
# seq_length = 1024
# micro_batch_size = train_loader.batch_size  # 1
# grad_accum_steps = 8
num_epochs = 2
debug_examples = 80  # 10 batches × batch_size 8

In [ ]:
train_config = TrainConfig(
    num_steps=(num_epochs * len(train_loader.loader)) // grad_accum_steps,
    learning_rate=1e-5,
    total_batch_size=micro_batch_size * seq_length * grad_accum_steps,
    micro_batch_size=micro_batch_size,
    seq_length=seq_length,
    accumulation_steps=grad_accum_steps,
    max_lr=1e-5,
    min_lr=1e-5,
    warmup_steps=0,
    weight_decay=0.1,
    device=device,
    use_compile=False,
    precision="fp32",
    )

In [ ]:
torch.manual_seed(123)

In [ ]:
def train(text):
    accelerator = Accelerator(mixed_precision="fp16")
    
    
    config =load_model_config(dataset_max_len=1024)
    model = return_model(MODEL_NAME)
    # model = GPT.from_pretrained(config, "gpt2")
    # print(f"Model Loaded: {model}")
    # model.eval()
    # model = model.to(device)
    
    model.eval()
    print("Testing Model with Basic Inference:")
    
    in_tokens = text_to_token_ids(text, tokenizer=tokenizer, 
                                #   allowed_special = {"<|endoftext|>"}
    )
    in_tokens = in_tokens.to(next(model.parameters()).device)
    print(in_tokens.shape)  # [1, sequence_length]
    out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
    print(token_ids_to_text(out_tokens, tokenizer))
    
    base_loader = train_loader.loader

    small_loader = DataLoader(
        Subset(base_loader.dataset, range(debug_examples)),
        batch_size=base_loader.batch_size,
        collate_fn=base_loader.collate_fn,
        shuffle=True,
        drop_last=True,
        num_workers=base_loader.num_workers,
    )

    train_loader_subset = CyclingDataLoader(small_loader,
        seq_len=train_loader.seq_len,
    )

    optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)
    
    train_comp = TrainComponents(optimizer=optimizer, 
                            dataloader=train_loader, 
                            scheduler= cosine_lr, 
                            forward_step=sft_forward,
                            loss_fn=sft_loss
                            )
    torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 
    print(train_config.accumulation_steps)
    


    run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

    logger = ExperimentLogger(
        project="sft-experiments",
        log_dir=str(project_root / "execution_results" / "runs" / run_name))

    model = train_model(model, train_components=train_comp, train_config=train_config, logger = logger)
    return model

## 5. Inspect saved artifacts
The script saves the model and tokenizer on successful completion.


In [ ]:
# text_1 = "What is your name ? "
text_1 = "Every effort moves you ? What is this"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
)
in_tokens=in_tokens.to(device)
print(in_tokens.shape)  # [1, sequence_length]

In [ ]:


out_tokens = gen_sample(model, in_tokens, max_new_tokens=100, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

## Notes

- The training script uses the existing instruction collator. Its Qwen padding and end-of-message behavior should be verified before relying on a full run.
- The script currently reports the last batch loss each epoch; it does not evaluate validation loss.
- Check the output directory before rerunning if you need to preserve a previous model.
